# Pipeline 3 – Silver ➜ Gold: the "Moneyball" Scouting Matrix (full rebuild)

Joins tournament **performance** (StatsBomb, Pipeline 1) with **market value** (Transfermarkt, Pipeline 2).

**Steps**
1. **Identity resolution** – StatsBomb and Transfermarkt share no key. Players are blocked by nationality
   (national team ⇔ citizenship) and matched with Levenshtein similarity + token containment on
   accent-normalised names, with a global exact-name fallback for dual nationals.
2. **Point-in-time valuation** – value *before* the tournament (what a club would have paid) and
   12 months *after* it (what the player turned out to be worth), using window functions.
3. **Performance score** – position-weighted percentiles of per-90 metrics (min. 180 minutes).
4. **Segmentation** – *Moneyball Target* = top-20 % performer in their position, below-median price tag, aged ≤ 26.
5. **Back-test** – did Moneyball Targets actually gain value in the 12 months after the tournament?

**Output (gold, Parquet + CSV export for Excel / QuickSight):** `player_identity_bridge`,
`moneyball_scouting_matrix`, `moneyball_backtest`, `pipeline_quality`.

Not incremental by design: the gold tables are small and the percentiles/medians depend on the whole
population, so the job rebuilds them in full (overwrite) on every run.

In [ ]:
%idle_timeout 30
%glue_version 5.0
%worker_type G.1X
%number_of_workers 2

In [ ]:
# --- Glue boilerplate: Spark + Glue contexts and job parameters ---
import sys
import time
import json
from awsglue.utils import getResolvedOptions
from pyspark.context import SparkContext
from awsglue.context import GlueContext
from awsglue.job import Job

sc = SparkContext.getOrCreate()
glueContext = GlueContext(sc)
spark = glueContext.spark_session
job = Job(glueContext)

# Job parameters are passed by the Workflow trigger (--BUCKET ...); interactive runs fall back to defaults
try:
    args = getResolvedOptions(sys.argv, ["JOB_NAME", "BUCKET"])
except Exception:
    args = {"JOB_NAME": "pl_03_silver_to_gold_moneyball", "BUCKET": "football-analytics-moneyball-<ACCOUNT_ID>"}
job.init(args["JOB_NAME"], args)

In [ ]:
# --- Configuration ---
from pyspark.sql import functions as F
from pyspark.sql.window import Window

SILVER = f"s3://{args['BUCKET']}/silver"
GOLD = f"s3://{args['BUCKET']}/gold"
PIPELINE = "pl_03_gold_moneyball"

MIN_MINUTES = 180            # sample-size floor for per-90 metrics (2 full matches)
MATCH_THRESHOLD = 0.85       # minimum similarity score to accept a name match
MATCH_MARGIN = 0.05          # best candidate must beat the runner-up by this much
TOP_PERFORMER_PCT = 0.80     # performance percentile that counts as "top 20 %"
MAX_TARGET_AGE = 26          # v2 rule: the market (rightly) discounts older players, so targets must be <= 26

# Position-specific weights for the composite performance score (each group sums to 1)
WEIGHTS = {
    "ATT": {"np_xg_p90": .25, "xa_p90": .20, "box_entries_p90": .15, "successful_dribbles_p90": .15,
            "progressive_carries_p90": .10, "final_third_pressures_p90": .15},
    "MID": {"progressive_passes_p90": .25, "xa_p90": .15, "progressive_carries_p90": .15,
            "box_entries_p90": .15, "pressures_p90": .15, "defensive_actions_p90": .15},
    "DEF": {"defensive_actions_p90": .30, "progressive_passes_p90": .25, "progressive_carries_p90": .15,
            "pressures_p90": .15, "pass_completion_pct": .15},
}
RUN_START = time.time()

In [ ]:
# --- Helpers: storage I/O + observability ---
def write_table(df, path, partition_cols, mode):
    """Write a Parquet table (snappy), optionally partitioned."""
    df.write.mode(mode).partitionBy(*partition_cols).parquet(path)


def write_csv(df, path):
    """Single CSV file with header – the stakeholder-friendly export (Excel, QuickSight, Athena)."""
    df.coalesce(1).write.mode("overwrite").option("header", True).csv(path)


def emit_metrics(metrics):
    """Structured log line (lands in CloudWatch Logs) + custom CloudWatch metrics."""
    print(json.dumps({"pipeline": PIPELINE, **metrics}))
    try:
        import boto3
        boto3.client("cloudwatch").put_metric_data(
            Namespace="FootballAnalytics/Pipelines",
            MetricData=[
                {"MetricName": k, "Value": float(v), "Unit": "Count",
                 "Dimensions": [{"Name": "Pipeline", "Value": PIPELINE}]}
                for k, v in metrics.items() if isinstance(v, (int, float))
            ],
        )
    except Exception as exc:  # metrics must never fail the pipeline
        print(f"CloudWatch metrics not published: {exc}")


def normalise(col):
    """Same accent/punctuation normalisation as Pipeline 2 so both sides are comparable."""
    accents = "áàâãäåāăąçćčďđéèêëēėęěğíìîïıīłńñňóòôõöøōőřśšşșťțúùûüůūűýÿžźżß"
    plain   = "aaaaaaaaacccddeeeeeeeegiiiiiilnnnoooooooorssssttuuuuuuuyyzzzs"
    return F.trim(F.regexp_replace(F.regexp_replace(
        F.translate(F.lower(col), accents, plain), r"[^a-z ]", " "), r"\s+", " "))

## 1. Load silver inputs
Transfermarkt silver keeps every snapshot – the gold layer always uses the most recent one.

In [ ]:
stats = spark.read.parquet(f"{SILVER}/statsbomb/player_season_stats")
matches = spark.read.parquet(f"{SILVER}/statsbomb/matches")

tm_players_all = spark.read.parquet(f"{SILVER}/transfermarkt/players")
latest_snapshot = tm_players_all.agg(F.max("snapshot_date")).first()[0]
tm_players = tm_players_all.where(F.col("snapshot_date") == latest_snapshot)
tm_vals = spark.read.parquet(f"{SILVER}/transfermarkt/player_valuations") \
    .where(F.col("snapshot_date") == latest_snapshot)
print(f"Transfermarkt snapshot used: {latest_snapshot}")

# Tournament window per competition (first and last match date)
tournaments = matches.groupBy("competition_id", "season_id", "competition_name", "season_name").agg(
    F.min("match_date").alias("tournament_start"), F.max("match_date").alias("tournament_end"))

## 2. Identity resolution (the hard part)
Candidate pairs are generated only *within the same nationality* (blocking) – this keeps the
comparison space small (~25 squad players × a few hundred active pros per country) and removes
most homonyms. Each pair gets a similarity score; the best candidate wins if it is confident enough.

In [ ]:
sb = (stats.select("player_id", "player_name", "player_nickname", "team").distinct()
      .withColumn("sb_name", normalise(F.col("player_name")))
      .withColumn("sb_nick", normalise(F.col("player_nickname")))
      # Tokens from the full legal name AND the nickname ("Daniel Olmo Carvajal" + "Daniel Olmo")
      .withColumn("sb_tokens", F.array_union(F.split("sb_name", " "),
                                             F.coalesce(F.split("sb_nick", " "), F.array().cast("array<string>"))))
      .withColumn("sb_first", F.coalesce(F.split("sb_nick", " ")[0], F.split("sb_name", " ")[0])))

# Plausible candidates = active around the tournaments. `last_season` only counts seasons in leagues the
# dataset covers (a player in Slovakia's league shows 2021), so a recent valuation also counts as active.
recently_valued = (tm_vals.where(F.col("valuation_date") >= "2023-01-01")
                   .select("player_id").distinct().withColumn("recently_valued", F.lit(True)))
tm = (tm_players.join(recently_valued, "player_id", "left")
      .where((F.coalesce(F.col("last_season"), F.lit(9999)) >= 2022) | F.col("recently_valued").isNotNull())
      .select(F.col("player_id").alias("tm_player_id"), F.col("player_name").alias("tm_name"),
              "country_of_citizenship", F.col("name_normalised").alias("tm_norm"))
      .withColumn("tm_norm", F.regexp_replace("tm_norm", r"\s+", " "))
      .withColumn("tm_tokens", F.split("tm_norm", " ")))


def lev_sim(a, b):
    """1 - normalised Levenshtein distance (1.0 = identical strings)."""
    return 1 - F.levenshtein(a, b) / F.greatest(F.length(a), F.length(b))


def score_pairs(pairs):
    """Similarity = best of four signals:
    * full-name / nickname Levenshtein  -> spelling variants ('Mykhailo' vs 'Mykhaylo', 'Malinovskiy' vs 'Malinovskyi')
    * token containment (both ways)     -> 'Vinicius Junior' inside 'Vinicius Jose Paixao de Oliveira Junior'
    * surname + first-name prefix        -> 'Dani Olmo' vs 'Daniel Olmo', 'Dayot' vs 'Dayotchanculle Upamecano'"""
    common = F.size(F.array_intersect("tm_tokens", "sb_tokens"))
    shortest = F.least(F.size("tm_tokens"), F.size("sb_tokens"))
    containment = F.when(shortest >= 2, common / shortest * 0.97)
    tm_first, tm_last = F.col("tm_tokens")[0], F.element_at("tm_tokens", -1)
    prefix = F.when((F.size("tm_tokens") >= 2) & F.array_contains("sb_tokens", tm_last)
                    & (F.col("sb_first").startswith(tm_first) | tm_first.startswith(F.col("sb_first"))), 0.92)
    return (pairs
            .withColumn("s_name", lev_sim(F.col("sb_name"), F.col("tm_norm")))
            .withColumn("s_nick", F.coalesce(lev_sim(F.col("sb_nick"), F.col("tm_norm")), F.lit(0.0)))
            .withColumn("s_tokens", F.coalesce(containment, F.lit(0.0)))
            .withColumn("s_prefix", F.coalesce(prefix, F.lit(0.0)))
            .withColumn("score", F.greatest("s_name", "s_nick", "s_tokens", "s_prefix"))
            .withColumn("method",
                        F.when(F.col("s_name") == 1, "exact_name")
                         .when(F.col("s_nick") == 1, "exact_nickname")
                         .when(F.col("score") == F.col("s_tokens"), "token_containment")
                         .when(F.col("score") == F.col("s_prefix"), "surname_first_prefix")
                         .otherwise("levenshtein")))


# Pass 1: blocked by nationality
pairs = score_pairs(sb.join(tm, sb.team == tm.country_of_citizenship))

w_sb = Window.partitionBy("player_id").orderBy(F.desc("score"))
ranked = (pairs.withColumn("rank", F.row_number().over(w_sb))
          .withColumn("runner_up", F.lead("score").over(w_sb)))
best = ranked.where((F.col("rank") == 1) & (F.col("score") >= MATCH_THRESHOLD)
                    & (F.coalesce(F.col("score") - F.col("runner_up"), F.lit(1.0)) >= MATCH_MARGIN))

# Pass 2 (fallback): dual nationals whose Transfermarkt citizenship differs from their national team –
# accept only an exact normalised name that is unique across the whole player base
unmatched = sb.join(best.select("player_id"), "player_id", "left_anti")
exact = (unmatched.join(tm, (unmatched.sb_name == tm.tm_norm) | (unmatched.sb_nick == tm.tm_norm))
         .withColumn("n", F.count("*").over(Window.partitionBy("player_id")))
         .where("n = 1").drop("n"))
fallback = score_pairs(exact).withColumn("method", F.lit("global_exact"))

cols = ["player_id", "player_name", "team", "tm_player_id", "tm_name", "score", "method"]
bridge = best.select(*cols).unionByName(fallback.select(*cols))

# Enforce 1:1 – if two StatsBomb players claim the same Transfermarkt id keep the higher score
bridge = (bridge.withColumn("r", F.row_number().over(Window.partitionBy("tm_player_id").orderBy(F.desc("score"))))
          .where("r = 1").drop("r")
          .withColumnRenamed("player_id", "sb_player_id")
          .withColumn("score", F.round("score", 3))).cache()

n_sb = sb.count()
n_matched = bridge.count()
print(f"Matched {n_matched}/{n_sb} StatsBomb players ({n_matched / n_sb:.1%})")
bridge.groupBy("method").count().orderBy(F.desc("count")).show()

## 3. Point-in-time market values (window functions)
* **pre-tournament value** – latest valuation *before* the first match (the buy price).
* **value 12 months after** – latest valuation up to 1 year after the final (the back-test).
* **latest value** – most recent valuation in the snapshot.

In [ ]:
def value_as_of(cond_col, alias):
    """Latest valuation satisfying a date condition, per player and tournament."""
    w = Window.partitionBy("tm_player_id", "competition_id").orderBy(F.desc("valuation_date"))
    return (pv.where(cond_col).withColumn("rn", F.row_number().over(w)).where("rn = 1")
            .select("tm_player_id", "competition_id",
                    F.col("market_value_eur").alias(alias),
                    F.col("valuation_date").alias(f"{alias}_date")))


pv = (tm_vals.select(F.col("player_id").alias("tm_player_id"), "valuation_date", "market_value_eur",
                     "value_12m_ago_eur")
      .crossJoin(F.broadcast(tournaments.select("competition_id", "tournament_start", "tournament_end"))))

pre = value_as_of(F.col("valuation_date") < F.col("tournament_start"), "value_pre_tournament_eur")
post = value_as_of((F.col("valuation_date") > F.col("tournament_end"))
                   & (F.col("valuation_date") <= F.date_add("tournament_end", 365)), "value_12m_after_eur")
latest = (tm_vals.where("is_latest").select(F.col("player_id").alias("tm_player_id"),
                                            F.col("market_value_eur").alias("value_latest_eur"),
                                            F.col("valuation_date").alias("value_latest_date")))

# Year-on-year trend going into the tournament (from Pipeline 2's window feature)
trend = value_as_of(F.col("valuation_date") < F.col("tournament_start"), "_v") \
    .join(pv.select("tm_player_id", "competition_id", F.col("valuation_date").alias("_v_date"),
                    "value_12m_ago_eur"), ["tm_player_id", "competition_id", "_v_date"]) \
    .select("tm_player_id", "competition_id", "value_12m_ago_eur")

## 4. Performance score – position-weighted percentiles
Every metric is turned into a percentile *within the position group* across both tournaments, then
combined with the weights above into a 0–100 score. Goalkeepers are excluded (no GK-specific metrics).

In [ ]:
eligible = (stats.where((F.col("minutes") >= MIN_MINUTES) & (F.col("position_group") != "GK"))
            .join(tournaments.select("competition_id", "season_id", "tournament_start", "tournament_end"),
                  ["competition_id", "season_id"]))

w_grp = Window.partitionBy("position_group")
metric_cols = sorted({m for w in WEIGHTS.values() for m in w})
for m in metric_cols:
    eligible = eligible.withColumn(f"pct_{m}", F.percent_rank().over(w_grp.orderBy(m)))

score_expr = None
for grp, weights in WEIGHTS.items():
    expr = sum(F.col(f"pct_{m}") * wt for m, wt in weights.items())
    score_expr = (F.when(F.col("position_group") == grp, expr) if score_expr is None
                  else score_expr.when(F.col("position_group") == grp, expr))
eligible = eligible.withColumn("performance_score", F.round(score_expr * 100, 1))
eligible = eligible.withColumn("performance_pct", F.percent_rank().over(w_grp.orderBy("performance_score")))

## 5. Assemble the scouting matrix + Moneyball segmentation

In [ ]:
matrix = (eligible.join(bridge.select("sb_player_id", "tm_player_id", F.col("score").alias("match_score"),
                                      F.col("method").alias("match_method")),
                        eligible.player_id == bridge.sb_player_id, "left")
          .join(tm_players.select(F.col("player_id").alias("tm_player_id"), "date_of_birth", "current_club_name",
                                  "current_competition_id", "contract_expiration_date", "sub_position", "foot",
                                  "agent_name", "url"), "tm_player_id", "left")
          .join(pre, ["tm_player_id", "competition_id"], "left")
          .join(post, ["tm_player_id", "competition_id"], "left")
          .join(trend, ["tm_player_id", "competition_id"], "left")
          .join(latest, "tm_player_id", "left")
          .join(tournaments.select("competition_id", "competition_name", "season_name"), "competition_id"))

matrix = (matrix
          .withColumn("age_at_tournament", F.floor(F.months_between("tournament_start", "date_of_birth") / 12))
          .withColumn("value_change_12m_pct",
                      F.round((F.col("value_12m_after_eur") / F.col("value_pre_tournament_eur") - 1) * 100, 1))
          .withColumn("value_trend_pre_12m_pct",
                      F.round((F.col("value_pre_tournament_eur") / F.col("value_12m_ago_eur") - 1) * 100, 1))
          .withColumn("contract_years_left_now",
                      F.round(F.months_between("contract_expiration_date", F.lit(latest_snapshot)) / 12, 1)))

# Price position vs. peers: median pre-tournament value of matched players in the same position group
valued = matrix.where(F.col("value_pre_tournament_eur").isNotNull())
medians = valued.groupBy("position_group").agg(
    F.percentile_approx("value_pre_tournament_eur", 0.5).alias("position_median_value_eur"))
matrix = matrix.join(medians, "position_group", "left")

is_top = F.col("performance_pct") >= TOP_PERFORMER_PCT
is_cheap = F.col("value_pre_tournament_eur") <= F.col("position_median_value_eur")
is_young = F.col("age_at_tournament") <= MAX_TARGET_AGE
matrix = matrix.withColumn(
    "segment",
    F.when(F.col("value_pre_tournament_eur").isNull(), "Unvalued / unmatched")
     .when(is_top & is_cheap & is_young, "Moneyball Target")
     .when(is_top & is_cheap, "Veteran Value")
     .when(is_top, "Proven Star")
     .when(~is_top & ~is_cheap & (F.col("performance_pct") < 0.40), "Overpriced")
     .otherwise("Squad Player"))

# Value-for-money: performance points per log-euro (higher = more output per € spent)
matrix = matrix.withColumn(
    "value_for_money_index",
    F.round(F.col("performance_score") / F.log10(F.greatest(F.col("value_pre_tournament_eur"), F.lit(100000))) * 10, 1))

scouting_matrix = matrix.select(
    "competition_name", "season_name", F.col("player_id").alias("sb_player_id"), "tm_player_id",
    "player_name", "player_nickname", F.col("team").alias("national_team"), "position_group", "primary_position",
    "sub_position", "foot", "age_at_tournament", "current_club_name", "current_competition_id",
    "appearances", "minutes", "np_goals", "assists", "np_xg_p90", "xa_p90", "key_passes_p90",
    "progressive_passes_p90", "progressive_carries_p90", "box_entries_p90", "successful_dribbles_p90",
    "pressures_p90", "final_third_pressures_p90", "defensive_actions_p90", "pass_completion_pct",
    "performance_score", F.round("performance_pct", 3).alias("performance_pct"),
    "value_12m_ago_eur", "value_pre_tournament_eur", "value_pre_tournament_eur_date", "position_median_value_eur",
    "value_trend_pre_12m_pct", "value_12m_after_eur", "value_change_12m_pct", "value_latest_eur",
    "contract_expiration_date", "contract_years_left_now", "agent_name",
    "segment", "value_for_money_index", "match_score", "match_method", "url",
).cache()

## 6. Back-test – did the model find value?
For each segment: how did market value move in the 12 months after the tournament?

In [ ]:
backtest = (scouting_matrix.where(F.col("value_change_12m_pct").isNotNull())
            .groupBy("segment")
            .agg(F.count("*").alias("players"),
                 F.round(F.avg("performance_score"), 1).alias("avg_performance_score"),
                 F.percentile_approx("value_pre_tournament_eur", 0.5).alias("median_value_pre_eur"),
                 F.percentile_approx("value_12m_after_eur", 0.5).alias("median_value_12m_after_eur"),
                 F.percentile_approx("value_change_12m_pct", 0.5).alias("median_value_change_pct"),
                 F.round(F.avg(F.when(F.col("value_change_12m_pct") > 0, 1).otherwise(0)) * 100, 1)
                  .alias("pct_players_value_up"),
                 # "Portfolio" view: buy every player in the segment at the pre-tournament price
                 F.sum("value_pre_tournament_eur").alias("portfolio_cost_eur"),
                 F.sum("value_12m_after_eur").alias("portfolio_value_12m_after_eur"))
            .withColumn("portfolio_return_pct",
                        F.round((F.col("portfolio_value_12m_after_eur") / F.col("portfolio_cost_eur") - 1) * 100, 1))
            .orderBy(F.desc("median_value_change_pct")))
backtest.show(truncate=False)

## 7. Quality gates + write gold

In [ ]:
n_eligible = scouting_matrix.count()
n_valued = scouting_matrix.where(F.col("value_pre_tournament_eur").isNotNull()).count()
quality = {
    "sb_players": n_sb,
    "matched_players": n_matched,
    "match_rate_pct": round(n_matched / n_sb * 100, 1),
    "eligible_players": n_eligible,
    "eligible_with_value_pct": round(n_valued / max(n_eligible, 1) * 100, 1),
    "moneyball_targets": scouting_matrix.where(F.col("segment") == "Moneyball Target").count(),
}
# Guardrail: a sudden drop in match rate usually means an upstream schema/encoding change
assert quality["match_rate_pct"] >= 70, f"Identity match rate too low: {quality['match_rate_pct']}%"
emit_metrics(quality)

# One row per metric, built JVM-side (no Python worker needed)
quality_df = (spark.range(1)
              .select(F.explode(F.array(*[F.struct(F.lit(k).alias("metric"), F.lit(float(v)).alias("value"))
                                          for k, v in quality.items()])).alias("m"))
              .select("m.*")
              .withColumn("run_ts", F.current_timestamp())
              .withColumn("tm_snapshot", F.lit(str(latest_snapshot))))

write_table(bridge, f"{GOLD}/player_identity_bridge", [], "overwrite")
write_table(scouting_matrix, f"{GOLD}/moneyball_scouting_matrix", [], "overwrite")
write_table(backtest, f"{GOLD}/moneyball_backtest", [], "overwrite")
write_table(quality_df, f"{GOLD}/pipeline_quality", [], "overwrite")
write_csv(scouting_matrix, f"{GOLD}/exports/moneyball_scouting_matrix")
write_csv(backtest, f"{GOLD}/exports/moneyball_backtest")
write_csv(quality_df.select("metric", "value"), f"{GOLD}/exports/pipeline_quality")

In [ ]:
# --- Run summary + commit ---
emit_metrics({"duration_seconds": round(time.time() - RUN_START, 1)})
job.commit()